GPT

EJERCICIO 1

In [1]:
vehiculos = [
    {"patente": "AB123CD", "tipo": "camion", "anio": 2018},
    {"patente": "AC555XY", "tipo": "auto", "anio": 2022},
    {"patente": "BB999AA", "tipo": "moto", "anio": 2019},
    {"patente": "CD321BA", "tipo": "camion", "anio": 2015},
]

# Lambda para filtrar camiones
es_camion = lambda v: v["tipo"] == "camion"

# Comprensión de listas para obtener solo las patentes de camiones
patentes_camiones = [v["patente"] for v in vehiculos if es_camion(v)]

print("🚛 Patentes de camiones:", patentes_camiones)


🚛 Patentes de camiones: ['AB123CD', 'CD321BA']


In [2]:
class Vehiculo:
    def __init__(self, patente, tipo, anio):
        self.patente = patente
        self.tipo = tipo
        self.anio = anio

    def descripcion(self):
        return f"Vehículo {self.patente} - {self.tipo} ({self.anio})"


# Crear lista de instancias a partir de la lista original
instancias = [Vehiculo(v["patente"], v["tipo"], v["anio"]) for v in vehiculos]

for v in instancias:
    print(v.descripcion())


Vehículo AB123CD - camion (2018)
Vehículo AC555XY - auto (2022)
Vehículo BB999AA - moto (2019)
Vehículo CD321BA - camion (2015)


In [3]:
class Auto(Vehiculo):
    def info_extra(self):
        print(f"{self.patente}: Tiene 5 plazas.")

class Moto(Vehiculo):
    def info_extra(self):
        print(f"{self.patente}: Es ágil en la ciudad.")

class Camion(Vehiculo):
    def info_extra(self):
        print(f"{self.patente}: Gran capacidad de carga.")


# Ejemplo de uso
flota = [
    Auto("AC555XY", "auto", 2022),
    Moto("BB999AA", "moto", 2019),
    Camion("AB123CD", "camion", 2018),
]

print("\n💡 Información adicional de la flota:")
for veh in flota:
    veh.info_extra()



💡 Información adicional de la flota:
AC555XY: Tiene 5 plazas.
BB999AA: Es ágil en la ciudad.
AB123CD: Gran capacidad de carga.


In [6]:
from datetime import date

# Excepción personalizada
class VehiculoAntiguoError(Exception):
    pass


# Decorador
def verificar_antiguedad(func):
    def wrapper(*args, **kwargs):
        vehiculo = args[0]
        anio_actual = date.today().year
        antiguedad = anio_actual - vehiculo.anio
        if antiguedad > 10:
            raise VehiculoAntiguoError(
                f"❌ El vehículo {vehiculo.patente} tiene más de 10 años ({antiguedad} años)."
            )
        return func(*args, **kwargs)
    return wrapper


# Extender clase base
class VehiculoVerificado(Vehiculo):
    @verificar_antiguedad
    def registrar(self):
        print(f"✅ Vehículo {self.patente} registrado correctamente.")

# Ejemplo de uso:
try:
    v1 = VehiculoVerificado("AB123CD", "camion", 2018)
    v2 = VehiculoVerificado("CD321BA", "camion", 2010)
    v1.registrar()
    v2.registrar()
except VehiculoAntiguoError as e:
    print(e)


✅ Vehículo AB123CD registrado correctamente.
❌ El vehículo CD321BA tiene más de 10 años (15 años).


In [7]:
def guardar_vehiculos(vehiculos):
    try:
        with open("vehiculos.txt", "w", encoding="utf-8") as f:
            for v in vehiculos:
                f.write(f"{v.patente},{v.tipo},{v.anio}\n")
        print("📁 Vehículos guardados correctamente en 'vehiculos.txt'.")
    except Exception as e:
        print(f"❌ Error al guardar el archivo: {e}")

guardar_vehiculos(flota)

📁 Vehículos guardados correctamente en 'vehiculos.txt'.


EJERCICIO 2

In [ ]:
import requests
import time
import json

# Archivo donde se guardarán las respuestas
ARCHIVO_SALIDA = "iss.json"

# Lista para almacenar las respuestas
respuestas = []

# Realizar 5 solicitudes con pausa de 2 segundos
for i in range(5):
	try:
		print(f"Solicitud {i + 1}/5...")
		respuesta = requests.get("http://api.open-notify.org/iss-now.json", timeout=5)
		respuesta.raise_for_status()  # Lanza error si la respuesta no fue 200 OK
		datos = respuesta.json()
		respuestas.append(datos)
		print("✅ Éxito:", datos)
	except requests.exceptions.ConnectionError:
		print("❌ Error de conexión. No hay acceso a Internet.")
		respuestas.append({"error": "Error de conexión"})
	except requests.exceptions.Timeout:
		print("❌ La solicitud excedió el tiempo de espera.")
		respuestas.append({"error": "Timeout"})
	except requests.exceptions.RequestException as e:
		print("❌ Error en la solicitud:", e)
		respuestas.append({"error": str(e)})

	time.sleep(2)  # Espera 2 segundos antes de la próxima solicitud

# Guardar todas las respuestas en un archivo JSON
with open(ARCHIVO_SALIDA, "w", encoding="utf-8") as f:
	json.dump(respuestas, f, indent=4, ensure_ascii=False)

print(f"\n🛰️ Se guardaron {len(respuestas)} respuestas en '{ARCHIVO_SALIDA}'.")

In [ ]:
import json

ARCHIVO = "iss.json"

try:
    # Intentar abrir el archivo
    with open(ARCHIVO, "r", encoding="utf-8") as f:
        datos = json.load(f)
        print("✅ Archivo cargado correctamente.\n")

    # Procesar la lista de posiciones
    print("📍 Posiciones registradas:")
    for i, entrada in enumerate(datos, start=1):
        # Verificamos si hay un error o si es una posición válida
        if "iss_position" in entrada:
            lat = entrada["iss_position"]["latitude"]
            lon = entrada["iss_position"]["longitude"]
            print(f"{i}. Latitud: {lat}, Longitud: {lon}")
        else:
            print(f"{i}. ❌ Error en la entrada: {entrada.get('error', 'Desconocido')}")

except FileNotFoundError as e:
    print(f"❌ Error al leer el archivo: {e}")

except json.JSONDecodeError as e:
    print(f"❌ Error al decodificar el JSON: {e}")

except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")


In [ ]:
def filtrar_latitud(posiciones, umbral):
    """
    Generador que devuelve solo las posiciones cuya latitud absoluta
    sea mayor al umbral indicado.
    """
    for pos in posiciones:
        # Verificamos que sea una entrada válida (con 'iss_position')
        if "iss_position" in pos:
            try:
                lat = float(pos["iss_position"]["latitude"])
                if abs(lat) > umbral:
                    yield pos
            except (ValueError, KeyError):
                # Si hay datos inválidos, los ignoramos
                continue
            
# Ejemplo: filtrar posiciones donde la ISS está a más de 30° del ecuador
umbral = 7
posiciones_filtradas = list(filtrar_latitud(datos, umbral))

print(f"📊 Posiciones con |latitud| > {umbral}:")
for p in posiciones_filtradas:
    lat = p["iss_position"]["latitude"]
    lon = p["iss_position"]["longitude"]
    print(f"Latitud: {lat}, Longitud: {lon}")

In [ ]:
# Función lambda para convertir strings a float redondeado a 2 decimales
convertir = lambda x: round(float(x), 2)

# Comprensión de diccionario:
# clave → timestamp
# valor → tupla (latitud, longitud) ya convertidas y redondeadas
posiciones_dict = {
    pos["timestamp"]: (
        convertir(pos["iss_position"]["latitude"]),
        convertir(pos["iss_position"]["longitude"])
    )
    for pos in datos
    if "iss_position" in pos  # Ignorar entradas con error
}

# Mostrar el resultado
print("📘 Diccionario procesado:\n")
for ts, (lat, lon) in posiciones_dict.items():
    print(f"{ts}: (Lat: {lat}, Lon: {lon})")


In [ ]:
from statistics import mean

# --- Decorador ---
def encabezado_iss(func):
    def wrapper(*args, **kwargs):
        print("=" * 10, "REPORTE DE POSICIONES DE LA ISS", "=" * 10)
        func(*args, **kwargs)
        print("=" * 54)
    return wrapper


# --- Clase ---
class ReporteISS:
    def __init__(self, fecha, posiciones):
        """
        fecha: string con la fecha del reporte
        posiciones: diccionario {timestamp: (lat, lon)}
        """
        self.fecha = fecha
        self.posiciones = posiciones

    @encabezado_iss
    def resumen(self):
        """Imprime la latitud máxima, mínima y promedio."""
        if not self.posiciones:
            print("No hay posiciones para analizar.")
            return

        # Extraer solo las latitudes
        latitudes = [lat for _, (lat, _) in self.posiciones.items()]

        print(f"📅 Fecha del reporte: {self.fecha}")
        print(f"🌍 Latitud máxima: {max(latitudes):.2f}")
        print(f"🌎 Latitud mínima: {min(latitudes):.2f}")
        print(f"📈 Promedio de latitudes: {mean(latitudes):.2f}")

reporte = ReporteISS("2025-10-16", posiciones_dict)
reporte.resumen()